# 🇸🇩 التدريب العصبي التلقائي للهجة السودانية (One-Click Automated Fine-Tuning)
### تطوير أوزان نموذج XTTS-v2 ليتكلم باللكنة ومخارج الحروف السودانية الأصيلة
---
**تم التحديث لإلغاء الواجهة اليدوية المعقدة — التدريب والحفظ إلى Google Drive يتم الآن آلياً بالكامل:**
1. `▶` **الخلية 1**: تثبيت حزمة التدريب المتوافقة وربط Google Drive.
2. `▶` **الخلية 2**: تجهيز وتقسيم الـ **4,120+ عينة صوتية** تلقائياً للتدريب والتقييم.
3. `▶` **الخلية 3**: تشغيل التدريب المباشر (Auto-Train) وحفظ النموذج تلقائياً في Google Drive.
4. `▶` **الخلية 4**: تجربة وسماع الصوت السوداني المدرب!

## 1️⃣ الخلية الأولى: تثبيت حزمة التدريب وربط Google Drive
*(تستغرق دقيقة واحدة — انتظر ظهور علامة الصح الخضراء `✓`)*

In [ ]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'

!pip install -q coqui-tts pandas soundfile

from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

print('\n✅ تم تثبيت المحرك وربط Google Drive بنجاح!')

## 2️⃣ الخلية الثانية: تجهيز وتقسيم البيانات للتدريب والتقييم (Auto-Split)
*(تقرأ الـ 4,120+ عينة وتنشئ ملفات `metadata_train.csv` و `metadata_eval.csv` في مجلد المشروع)*

In [ ]:
import pandas as pd
import os

DATASET_DIR = '/content/drive/MyDrive/Sudanese_Voice_Project/fine_tuning_dataset'
METADATA_FILE = os.path.join(DATASET_DIR, 'metadata.csv')
TRAIN_CSV = os.path.join(DATASET_DIR, 'metadata_train.csv')
EVAL_CSV = os.path.join(DATASET_DIR, 'metadata_eval.csv')

df = pd.read_csv(METADATA_FILE, sep='|', on_bad_lines='skip').dropna()
# تصفية أي نصوص قصيرة جداً أو علامات موسيقية
df = df[df['text'].str.len() >= 8]

# خيار حجم التدريب: استخدام 8,000 عينة نقية وموزعة يعطي تدريباً فائق الجودة وسريعاً (~25 دقيقة)
# يمكنك تعيين MAX_SAMPLES = None لتدريب كامل الداتاسيت (55 ألف عينة)
MAX_SAMPLES = 8000
if MAX_SAMPLES and len(df) > MAX_SAMPLES:
    df = df.sample(n=MAX_SAMPLES, random_state=42).reset_index(drop=True)
else:
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

split_idx = int(len(df) * 0.9)
train_df = df.iloc[:split_idx]
eval_df = df.iloc[split_idx:]

train_df.to_csv(TRAIN_CSV, sep='|', index=False)
eval_df.to_csv(EVAL_CSV, sep='|', index=False)

print(f'📊 إجمالي العينات النقية المعتمدة للتدريب: {len(df):,} عينة سودانية!')
print(f'🏋️‍♂️ عينات التدريب (Train): {len(train_df):,}')
print(f'🧪 عينات الاختبار (Eval): {len(eval_df):,}')
display(train_df.head(5))


## 3️⃣ الخلية الثالثة: إطلاق التدريب التلقائي وحفظ النموذج في Drive مباشرة
*(ستبدأ العملية آلياً وتظهر لك خطوات التدريب وتناقص نسبة الخطأ، وتحفظ النموذج النهائي مباشرة داخل مجلد المشروع على Google Drive)*

In [ ]:
import os, glob, shutil
from trainer import Trainer, TrainerArgs
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.configs.xtts_config import XttsAudioConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig
from TTS.utils.manage import ModelManager

os.environ["COQUI_TOS_AGREED"] = "1"

DATASET_DIR = "/content/drive/MyDrive/Sudanese_Voice_Project/fine_tuning_dataset"
TRAIN_CSV = os.path.join(DATASET_DIR, "metadata_train.csv")
EVAL_CSV = os.path.join(DATASET_DIR, "metadata_eval.csv")
OUT_DRIVE_DIR = "/content/drive/MyDrive/Sudanese_Voice_Project/fine_tuned_model"
BASE_DIR = "/content/xtts_base_v2"
RUN_OUT = "/content/xtts_ft_run"
os.makedirs(OUT_DRIVE_DIR, exist_ok=True)
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(RUN_OUT, exist_ok=True)

# 1️⃣ تنزيل ملفات XTTS-v2 الأساسية مباشرة من HuggingFace
HF = "https://huggingface.co/coqui/XTTS-v2/resolve/main"
links = [
    f"{HF}/dvae.pth",
    f"{HF}/mel_stats.pth",
    f"{HF}/vocab.json",
    f"{HF}/model.pth",
    f"{HF}/config.json",
    f"{HF}/speakers_xtts.pth",
]
needed = [u for u in links if not os.path.isfile(os.path.join(BASE_DIR, os.path.basename(u)))]
if needed:
    print("⏳ جاري تنزيل أوزان XTTS-v2 الأساسية من HuggingFace...")
    ModelManager._download_model_files(needed, BASE_DIR, progress_bar=True)

DVAE_CHECKPOINT = os.path.join(BASE_DIR, "dvae.pth")
MEL_NORM_FILE   = os.path.join(BASE_DIR, "mel_stats.pth")
TOKENIZER_FILE  = os.path.join(BASE_DIR, "vocab.json")
XTTS_CHECKPOINT = os.path.join(BASE_DIR, "model.pth")
XTTS_CONFIG     = os.path.join(BASE_DIR, "config.json")
XTTS_SPEAKERS   = os.path.join(BASE_DIR, "speakers_xtts.pth")

# 2️⃣ إعداد الداتاسيت ومعاملات التدريب المتوافقة تماماً مع coqui-tts 0.28+
config_dataset = BaseDatasetConfig(
    formatter="coqui",
    dataset_name="sudanese_dialect",
    path=DATASET_DIR,
    meta_file_train=TRAIN_CSV,
    meta_file_val=EVAL_CSV,
    language="ar",
)

model_args = GPTArgs(
    max_conditioning_length=132300,
    min_conditioning_length=66150,
    debug_loading_failures=False,
    max_wav_length=255995,
    max_text_length=200,
    mel_norm_file=MEL_NORM_FILE,
    dvae_checkpoint=DVAE_CHECKPOINT,
    xtts_checkpoint=XTTS_CHECKPOINT,
    tokenizer_file=TOKENIZER_FILE,
    gpt_num_audio_tokens=1026,
    gpt_start_audio_token=1024,
    gpt_stop_audio_token=1025,
    gpt_use_masking_gt_prompt_approach=True,
    gpt_use_perceiver_resampler=True,
)

audio_config = XttsAudioConfig(sample_rate=22050, dvae_sample_rate=22050, output_sample_rate=24000)

config = GPTTrainerConfig(
    epochs=4,
    output_path=RUN_OUT,
    model_args=model_args,
    run_name="GPT_XTTS_Sudanese_FT",
    project_name="Sudanese_Voice_AI",
    run_description="Sudanese Dialect XTTS-v2 Fine-Tuning",
    datasets=[config_dataset],
    dashboard_logger="tensorboard",
    audio=audio_config,
    batch_size=2,
    batch_group_size=48,
    eval_batch_size=2,
    num_loader_workers=2,
    eval_split_max_size=256,
    print_step=50,
    plot_step=250,
    log_model_step=500,
    save_step=1000,
    save_n_checkpoints=1,
    save_checkpoints=True,
    print_eval=False,
    optimizer="AdamW",
    optimizer_wd_only_on_weights=True,
    optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
    lr=5e-06,
    lr_scheduler="MultiStepLR",
    lr_scheduler_params={"milestones": [50000 * 18, 150000 * 18, 300000 * 18], "gamma": 0.5, "last_epoch": -1},
    test_sentences=[],
)

# 3️⃣ بناء النموذج وتحميل العينات بالواجهة الرسمية الحديثة (coqui-tts 0.28+)
model = GPTTrainer(config)
train_samples, eval_samples = load_tts_samples(
    config,
    eval_split=True,
    eval_split_max_size=config.eval_split_max_size,
    eval_split_size=config.eval_split_size,
)
print(f"🚀 جاهز! تم تحميل {len(train_samples):,} عينة تدريب و {len(eval_samples):,} عينة اختبار — بدء التدريب على كرت الشاشة...")

trainer = Trainer(
    TrainerArgs(
        restore_path=None,
        skip_train_epoch=False,
        start_with_eval=False,
        grad_accum_steps=16,
    ),
    config,
    output_path=RUN_OUT,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)
trainer.fit()

# 4️⃣ نسخ أفضل نموذج مدرب مع ملفات التشغيل إلى Google Drive
candidates = glob.glob(os.path.join(trainer.output_path, "**", "best_model*.pth"), recursive=True)
if not candidates:
    candidates = glob.glob(os.path.join(trainer.output_path, "**", "checkpoint_*.pth"), recursive=True)
if candidates:
    best_ckpt = max(candidates, key=os.path.getctime)
    shutil.copy2(best_ckpt, os.path.join(OUT_DRIVE_DIR, "model.pth"))
    print(f"✅ تم حفظ أفضل أوزان سودانية ({os.path.basename(best_ckpt)}) إلى Google Drive!")
shutil.copy2(XTTS_CONFIG, os.path.join(OUT_DRIVE_DIR, "config.json"))
shutil.copy2(TOKENIZER_FILE, os.path.join(OUT_DRIVE_DIR, "vocab.json"))
shutil.copy2(XTTS_SPEAKERS, os.path.join(OUT_DRIVE_DIR, "speakers_xtts.pth"))
print("\n🎉 اكتمل التدريب العصبي وحُفظ النموذج السوداني المطور بنجاح في Google Drive!")


## 4️⃣ الخلية الرابعة: تجربة وسماع الصوت السوداني المدرب فوراً
*(يقرأ النموذج المدرب من مجلد `fine_tuned_model` في Drive ويولد الجملة بالنبرة السودانية الأصيلة)*

In [ ]:
import torch
from TTS.api import TTS
from IPython.display import Audio, display
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
FINE_TUNED_MODEL_DIR = '/content/drive/MyDrive/Sudanese_Voice_Project/fine_tuned_model'
REF_WAV = '/content/drive/MyDrive/Sudanese_Voice_Project/corpus/references/clean_podcast_8s.wav'

if os.path.exists(os.path.join(FINE_TUNED_MODEL_DIR, 'model.pth')):
    print('🎉 جاري تحميل النموذج السوداني المدرب حديثاً...')
    xtts_tuned = TTS(model_path=FINE_TUNED_MODEL_DIR, config_path=os.path.join(FINE_TUNED_MODEL_DIR, 'config.json')).to(device)
else:
    print('⚠️ لم يكتمل حفظ model.pth بعد، جاري استخدام النموذج الأساسي.')
    xtts_tuned = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

test_phrase = 'يا زول مرحب بيك في استوديو الصوت السوداني، دحين كدا النطق بقى سمح بالحيل وظابط تمام.'
out_audio = '/content/drive/MyDrive/Sudanese_Voice_Project/cloned_sudanese.wav'

xtts_tuned.tts_to_file(
    text=test_phrase,
    speaker_wav=REF_WAV,
    language='ar',
    file_path=out_audio,
    temperature=0.6,
    repetition_penalty=2.0
)

print('✅ تم التوليد والحفظ في Drive بنجاح!')
display(Audio(out_audio, autoplay=True))
